In [1]:
%cd /content/drive/MyDrive/ResearchProjects/MedCLIP/Scripts
from config import  DATA_DIR,RESULTS_DIR,FIGURS_DIR,METADATA_DIR
import pandas as pd
from pathlib import Path
import numpy as np
from util import preprocess_adni

/content/drive/MyDrive/ResearchProjects/MedCLIP/Scripts
pyreadstat is already installed ✅
optuna is already installed ✅
joblib is already installed ✅
dcurves is already installed ✅
python-dotenv not found. Installing...
python-dotenv installed successfully ✅
nibabel is already installed ✅
SimpleITK is already installed ✅
scipy is already installed ✅
deepbrain is already installed ✅
antspyx not found. Installing...
antspyx installed successfully ✅
nilearn is already installed ✅
antspynet is already installed ✅


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


monai is already installed ✅


In [2]:
df_processed = pd.read_csv(Path(RESULTS_DIR, "ADNI_Combined_Multimodal.csv"))

CLASSES = df_processed['Group'].unique().tolist()

if Path(RESULTS_DIR, "train_split.csv").exists():

    print("Loading existing train/val/test splits...")
    train_df = pd.read_csv(Path(RESULTS_DIR, "train_split.csv"))
    val_df   = pd.read_csv(Path(RESULTS_DIR, "val_split.csv"))
    test_df  = pd.read_csv(Path(RESULTS_DIR, "test_split.csv"))
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

    train_idx, test_idx = next(gss.split(df_processed, groups=df_processed["PTID"]))

    train_df = df_processed.iloc[train_idx]
    test_df  = df_processed.iloc[test_idx]

    # Further split train → train/val, again by subject
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    tr_idx, val_idx = next(gss_val.split(train_df, groups=train_df["PTID"]))

    val_df   = train_df.iloc[val_idx]
    train_df = train_df.iloc[tr_idx]

    # save the splits for future reference
    train_df.to_csv(Path(RESULTS_DIR, "train_split.csv"), index=False)
    val_df.to_csv(Path(RESULTS_DIR, "val_split.csv"), index=False)
    test_df.to_csv(Path(RESULTS_DIR, "test_split.csv"), index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# --- Preprocessing ---
data = preprocess_adni(train_df, val_df, test_df,keep_paths=False)

X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]

Loading existing train/val/test splits...
Train: 626 | Val: 102 | Test: 140
Splits — Train: 626 | Val: 102 | Test: 140
Dropping 15 column(s) exceeding 60% missingness: ['EcogSPVisspat_bl', 'EcogSPPlan_bl', 'EcogSPOrgan_bl', 'EcogSPDivatt_bl', 'EcogSPTotal_bl', 'EcogPtTotal_bl', 'EcogPtVisspat_bl', 'EcogPtLang_bl', 'EcogPtMem_bl', 'MOCA_bl', 'EcogPtOrgan_bl', 'EcogPtDivatt_bl', 'EcogSPLang_bl', 'EcogSPMem_bl', 'EcogPtPlan_bl']
Classes: ['AD' 'CN' 'LMCI']
Label distribution — Train: [100 233 293] | Val: [12 34 56] | Test: [28 37 75]
Feature matrix — Train: (626, 61) | Val: (102, 61) | Test: (140, 61)


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from config import NUMERIC_PREDICTORS, CATEGORICAL_PREDICTORS
# -------------------------------------------------------------------
# 1. Encode labels
# -------------------------------------------------------------------
le = LabelEncoder()                        # CN=0, MCI=1, AD=2
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

print("Classes:", le.classes_)
print("Train distribution:", np.bincount(y_train_enc))
print("Val   distribution:", np.bincount(y_val_enc))
print("Test  distribution:", np.bincount(y_test_enc))

# -------------------------------------------------------------------
# 2. Train
# -------------------------------------------------------------------
clf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",   # handles CN/MCI/AD imbalance
    max_features="sqrt",
    min_samples_leaf=5,        # mild regularisation
    random_state=42,
    n_jobs=-1,
)

clf.fit(X_train, y_train_enc)

# -------------------------------------------------------------------
# 3. Evaluate on val then test
# -------------------------------------------------------------------
def evaluate(model, X, y_true, le, split_name="Val"):
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)

    print(f"\n{'='*50}")
    print(f"  {split_name} Results")
    print(f"{'='*50}")
    print(f"Balanced Accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC (OvR macro): {roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro'):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))


    return y_pred, y_proba

val_pred,  val_proba  = evaluate(clf, X_val,  y_val_enc,  le, "Validation")
test_pred, test_proba = evaluate(clf, X_test, y_test_enc, le, "Test")



Classes: [0 1 2]
Train distribution: [100 233 293]
Val   distribution: [12 34 56]
Test  distribution: [28 37 75]

  Validation Results
Balanced Accuracy : 0.8595
ROC-AUC (OvR macro): 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.67      0.80        12
           1       1.00      0.91      0.95        34
           2       0.89      1.00      0.94        56

    accuracy                           0.93       102
   macro avg       0.96      0.86      0.90       102
weighted avg       0.94      0.93      0.93       102


  Test Results
Balanced Accuracy : 0.8271
ROC-AUC (OvR macro): 0.9612

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.75      0.74        28
           1       1.00      0.84      0.91        37
           2       0.84      0.89      0.86        75

    accuracy                           0.85       140
   macro avg       0.85      0.83     